# Alpha Sweep Evaluation

Noisy in-vivo 4개 샘플 (neuron2, neuron4, s06b, s10mm)에 대해:
- **Alpha sweep 결과**: F1/P/R vs ALPHA 그래프
- **Vaa3D 비교**: 각 metric에 수평 기준선
- **MIP 시각화**: Ours (best α) / Vaa3D / Gold Standard

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
import tifffile
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

ROOT      = Path('..').resolve()
SWEEP_DIR = ROOT / 'results' / 'alpha_sweep'
VAA3D_DIR = ROOT / 'results' / 'vaa3d'
GOLD_DIR  = ROOT / 'data' / 'gold_standard'
IMG_DIR   = ROOT / 'data' / 'images'

SAMPLES = [
    'neuron2',
    'neuron4',
    '1201_01_s06b_L36_Sum_ch2.tif',
    '1201_01_s10mm_ch2.tif',
]
SAMPLE_LABELS = {
    'neuron2': 'neuron2',
    'neuron4': 'neuron4',
    '1201_01_s06b_L36_Sum_ch2.tif': 's06b',
    '1201_01_s10mm_ch2.tif': 's10mm',
}

COLORS = {
    'ours':  '#4c9be8',
    'vaa3d': '#e87b4c',
    'gold':  '#f5c518',
}

## Helper functions

In [ ]:
def read_pixelsize(stem):
    for name in [f'{stem}.pixelsize.txt', f'{stem}.tif.pixelsize.txt']:
        p = GOLD_DIR / name
        if p.exists():
            m = re.search(r'Voxel size:\s*([\d.]+)x[\d.]+x([\d.]+)', p.read_text(), re.I)
            if m:
                return float(m.group(1)), float(m.group(2))
    return 1.0, 1.0


def load_swc(path, scale_xy=1.0, scale_z=1.0):
    nodes = {}
    for line in Path(path).read_text().splitlines():
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        if len(p) < 7: continue
        nid = int(p[0])
        nodes[nid] = dict(x=float(p[2])*scale_xy, y=float(p[3])*scale_xy,
                          z=float(p[4])*scale_z, pid=int(p[6]))
    return nodes


def pos_array(nodes):
    ids = sorted(nodes)
    arr = np.array([[nodes[i]['x'], nodes[i]['y'], nodes[i]['z']] for i in ids], dtype=np.float32)
    return arr


def compute_metrics(auto_nodes, gold_nodes, thr=2.0):
    a = pos_array(auto_nodes); g = pos_array(gold_nodes)
    if len(a) == 0:
        return dict(f1=0, precision=0, recall=0, esa=999)
    d_a2g, _ = cKDTree(g).query(a)
    d_g2a, _ = cKDTree(a).query(g)
    p  = float((d_a2g <= thr).mean())
    r  = float((d_g2a <= thr).mean())
    f1 = 2*p*r/(p+r+1e-8)
    return dict(f1=f1, precision=p, recall=r, esa=float((d_a2g.mean()+d_g2a.mean())/2))


def load_swc_segments(path, scale_xy=1.0, scale_z=1.0):
    nodes = load_swc(path, scale_xy, scale_z)
    segs = []
    for nid, n in nodes.items():
        if n['pid'] == -1 or n['pid'] not in nodes: continue
        p = nodes[n['pid']]
        segs.append((n['x'], n['y'], n['z'], p['x'], p['y'], p['z']))
    return segs


def make_lc(segs, proj, vxy, vz, color, lw=0.7, alpha=0.9):
    """segs: (x,y,z,x,y,z) µm 단위. vxy/vz로 픽셀 변환."""
    if not segs: return None
    if proj == 'xy':
        lines = [[(s[0]/vxy, s[1]/vxy), (s[3]/vxy, s[4]/vxy)] for s in segs]
    elif proj == 'xz':
        lines = [[(s[0]/vxy, s[2]/vz), (s[3]/vxy, s[5]/vz)] for s in segs]
    else:  # yz
        lines = [[(s[1]/vxy, s[2]/vz), (s[4]/vxy, s[5]/vz)] for s in segs]
    return LineCollection(lines, colors=color, linewidths=lw, alpha=alpha)


def vmax99(mip):
    pos = mip[mip > 0]
    return float(np.percentile(pos, 99)) if pos.size else 1.0

## 데이터 로드

In [ ]:
# sweep 결과 로드 — CSV 있으면 사용, 없으면 SWC 파일에서 직접 계산
# → sweep 중간에도 완료된 것만 보여줌

csv_path = SWEEP_DIR / 'sweep_results.csv'

# Gold standard 캐시 (vaa3d 계산에도 재사용)
gold_cache = {}
for stem in SAMPLES:
    vxy, vz = read_pixelsize(stem)
    g_path = GOLD_DIR / f'{stem}.swc'
    if g_path.exists():
        gold_cache[stem] = (load_swc(g_path, vxy, vz), vxy, vz)

if csv_path.exists():
    df_sweep = pd.read_csv(csv_path)
    print(f'CSV 로드: {len(df_sweep)}행')
else:
    # alpha_X.0/ 폴더 스캔 → SWC에서 직접 메트릭 계산
    print('sweep_results.csv 없음 → SWC 파일 직접 스캔 중...')
    rows = []
    for alpha_dir in sorted(SWEEP_DIR.glob('alpha_*/')):
        try:
            alpha = float(alpha_dir.name.replace('alpha_', ''))
        except ValueError:
            continue
        for stem in SAMPLES:
            swc_path = alpha_dir / f'{stem}.swc'
            if not swc_path.exists() or stem not in gold_cache:
                continue
            gold_nodes, vxy, vz = gold_cache[stem]
            auto_nodes = load_swc(swc_path)  # ours는 이미 µm
            m = compute_metrics(auto_nodes, gold_nodes)
            rows.append(dict(alpha=alpha, stem=stem, **m, n_auto=len(auto_nodes)))
            print(f'  alpha={alpha:.1f}  {SAMPLE_LABELS[stem]:8s}'
                  f'  F1={m["f1"]:.3f}  P={m["precision"]:.3f}'
                  f'  R={m["recall"]:.3f}  ESA={m["esa"]:.2f}')
    df_sweep = pd.DataFrame(rows)
    # sweep_alpha.py CSV의 컬럼명(sample)과 맞추기
    if 'stem' in df_sweep.columns:
        df_sweep = df_sweep.rename(columns={'stem': 'sample'})

print(f'\n완료: alpha={sorted(df_sweep["alpha"].unique())}  샘플={df_sweep["sample"].unique().tolist()}')

# Vaa3D 메트릭
vaa3d_metrics = {}
for stem in SAMPLES:
    v_path = VAA3D_DIR / f'{stem}.swc'
    if stem not in gold_cache or not v_path.exists():
        print(f'[SKIP] {stem}: vaa3d SWC 없음')
        continue
    gold_nodes, vxy, vz = gold_cache[stem]
    vaa3d_nodes = load_swc(v_path, vxy, vz)
    vaa3d_metrics[stem] = compute_metrics(vaa3d_nodes, gold_nodes)
    print(f'{SAMPLE_LABELS[stem]:8s}  Vaa3D'
          f'  F1={vaa3d_metrics[stem]["f1"]:.3f}'
          f'  P={vaa3d_metrics[stem]["precision"]:.3f}'
          f'  R={vaa3d_metrics[stem]["recall"]:.3f}'
          f'  ESA={vaa3d_metrics[stem]["esa"]:.2f}')

## Alpha Sweep 그래프

F1 / Precision / Recall vs ALPHA. Vaa3D는 점선 수평선.

In [ ]:
metrics = ['f1', 'precision', 'recall', 'esa']
metric_labels = ['F1', 'Precision', 'Recall', 'ESA (µm)']
n_samples = len(SAMPLES)

fig, axes = plt.subplots(n_samples, 4, figsize=(16, 3.5 * n_samples),
                          gridspec_kw={'hspace': 0.45, 'wspace': 0.3})

for row, stem in enumerate(SAMPLES):
    sub = df_sweep[df_sweep['sample'] == stem].sort_values('alpha')
    label = SAMPLE_LABELS[stem]
    vm = vaa3d_metrics.get(stem, {})

    for col, (metric, mlabel) in enumerate(zip(metrics, metric_labels)):
        ax = axes[row, col]
        if not sub.empty:
            ax.plot(sub['alpha'], sub[metric], 'o-', color=COLORS['ours'],
                    lw=2, ms=6, label='Ours')
            best_idx = sub[metric].idxmin() if metric == 'esa' else sub[metric].idxmax()
            brow = sub.loc[best_idx]
            ax.scatter([brow['alpha']], [brow[metric]], s=120, zorder=5,
                       color=COLORS['ours'], edgecolors='white', linewidths=1.5)

        if vm:
            ax.axhline(vm[metric], color=COLORS['vaa3d'], lw=1.5,
                       linestyle='--', label='Vaa3D')

        if col == 0:
            ax.set_ylabel(label, fontsize=11, fontweight='bold')
        ax.set_xlabel('ALPHA', fontsize=9)
        ax.set_title(mlabel, fontsize=10)
        if col == 3:
            ax.invert_yaxis()
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.suptitle('Alpha Sweep — Noisy In-vivo Samples (2-µm threshold)', fontsize=13, y=1.01)
plt.savefig('alpha_sweep_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: alpha_sweep_metrics.png')

## 요약 테이블: Best Alpha vs Vaa3D

In [ ]:
rows = []
for stem in SAMPLES:
    label = SAMPLE_LABELS[stem]
    sub = df_sweep[df_sweep['sample'] == stem]
    if not sub.empty:
        best = sub.loc[sub['f1'].idxmax()]
        rows.append(dict(sample=label, method=f'Ours (α={best["alpha"]:.1f})',
                         f1=best['f1'], precision=best['precision'],
                         recall=best['recall'], esa=best['esa']))
        cur = sub[sub['alpha'] == 9.0]
        if not cur.empty:
            c = cur.iloc[0]
            rows.append(dict(sample=label, method='Ours (α=9.0, 현재)',
                             f1=c['f1'], precision=c['precision'],
                             recall=c['recall'], esa=c['esa']))
    vm = vaa3d_metrics.get(stem)
    if vm:
        rows.append(dict(sample=label, method='Vaa3D', **vm))

df_table = pd.DataFrame(rows)
df_table[['f1','precision','recall','esa']] = df_table[['f1','precision','recall','esa']].round(3)

df_table.style \
    .background_gradient(cmap='RdYlGn',   subset=['f1','precision','recall']) \
    .background_gradient(cmap='RdYlGn_r', subset=['esa']) \
    .format({'f1':'{:.3f}','precision':'{:.3f}','recall':'{:.3f}','esa':'{:.2f}'})

## MIP 시각화

각 샘플: **Ours (best α)** / **Vaa3D** / **Gold Standard** — XY · XZ · YZ

In [ ]:
for stem in SAMPLES:
    label = SAMPLE_LABELS[stem]
    vxy, vz = read_pixelsize(stem)

    img_path = IMG_DIR / f'{stem}.tif'
    if not img_path.exists():
        print(f'[SKIP] 이미지 없음: {img_path}')
        continue

    sub = df_sweep[df_sweep['sample'] == stem]
    if sub.empty:
        print(f'[SKIP] sweep 결과 없음: {stem}')
        continue
    best_alpha = sub.loc[sub['f1'].idxmax(), 'alpha']
    ours_swc  = SWEEP_DIR / f'alpha_{best_alpha:.1f}' / f'{stem}.swc'
    gold_swc  = GOLD_DIR  / f'{stem}.swc'
    vaa3d_swc = VAA3D_DIR / f'{stem}.swc'

    methods = []
    if ours_swc.exists():
        methods.append((f'Ours (α={best_alpha:.1f})',
                        load_swc_segments(ours_swc), COLORS['ours']))  # 이미 µm
    if vaa3d_swc.exists():
        methods.append(('Vaa3D',
                        load_swc_segments(vaa3d_swc, vxy, vz), COLORS['vaa3d']))
    if gold_swc.exists():
        methods.append(('Gold',
                        load_swc_segments(gold_swc, vxy, vz), COLORS['gold']))

    print(f'Loading {img_path.name} ...')
    stack = tifffile.imread(str(img_path)).astype(np.float32)
    if stack.ndim == 4:
        stack = stack[0] if stack.shape[0] < stack.shape[1] else stack[:, 0]
    mip_xy = stack.max(axis=0)  # (Y, X)
    mip_xz = stack.max(axis=1)  # (Z, X)
    mip_yz = stack.max(axis=2)  # (Z, Y)
    mips  = [mip_xy, mip_xz, mip_yz]
    projs = ['xy', 'xz', 'yz']
    proj_titles = ['XY (Z-MIP)', 'XZ (Y-MIP)', 'YZ (X-MIP)']

    n_rows = 1 + len(methods)
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, 3.8 * n_rows),
                              gridspec_kw={'hspace': 0.05, 'wspace': 0.04})

    for col, (mip, title) in enumerate(zip(mips, proj_titles)):
        ax = axes[0, col]
        ax.imshow(mip, cmap='gray', origin='upper',
                  aspect='auto' if col > 0 else 'equal', vmax=vmax99(mip))
        ax.set_title(title, fontsize=10)
        if col == 0: ax.set_ylabel('Image', fontsize=10)
        ax.axis('off')

    for mrow, (mlabel, segs, color) in enumerate(methods, start=1):
        for col, (mip, proj) in enumerate(zip(mips, projs)):
            ax = axes[mrow, col]
            ax.imshow(mip, cmap='gray', origin='upper',
                      aspect='auto' if col > 0 else 'equal', vmax=vmax99(mip))
            lc = make_lc(segs, proj, vxy, vz, color)  # vz 추가
            if lc is not None:
                ax.add_collection(lc)
            if col == 0:
                ax.set_ylabel(mlabel, fontsize=10, color=color)
            ax.axis('off')

    best_f1 = sub.loc[sub['f1'].idxmax(), 'f1']
    plt.suptitle(f'{label}  |  best α={best_alpha:.1f}  F1={best_f1:.3f}'
                 f'  (Vaa3D F1={vaa3d_metrics.get(stem, {}).get("f1", 0):.3f})',
                 fontsize=12, y=1.005)
    plt.tight_layout()
    fname = f'mip_sweep_{label}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')